# 📚 The Bookstore Intelligence Mission
**Business Data and Text Mining — Assignment 2: Web Scraping**

This notebook scrapes the competitor catalogue at [books.toscrape.com](https://books.toscrape.com/), turns it into a clean
business dataset (`competitor_books.csv`), answers the manager's intelligence questions and solves the CEO's £100 challenge.

**How to run:** `Runtime → Run all`. Every result in this notebook is produced by the scraper; nothing is typed in by hand.

## Mission 1 — Reconnaissance

Each book on a catalogue page sits inside an `<article class="product_pod">` card. Inspecting one card gives:

| Data Needed | HTML Tag / Class Found | Notes |
|---|---|---|
| Book Title | `<h3>` → `<a title="...">` | The visible link text is truncated ("A Light in the ..."), so the **full title is read from the `title` attribute**. |
| Price | `<p class="price_color">` | Text such as `£51.77`; the `£` sign is stripped and the value converted to a number. |
| Rating | `<p class="star-rating Three">` | The rating is stored as the **second class word** (One–Five), which is mapped to 1–5. |
| Availability | `<p class="instock availability">` | Text "In stock" surrounded by whitespace and an icon, so it is stripped. |
| Product URL *(Bonus 2)* | `<h3>` → `<a href="...">` | Relative link, converted to a full URL with `urljoin`. |
| Next page | `<li class="next">` → `<a href="page-2.html">` | Used to move through the catalogue until there is no "next" button. |

## Setup

In [ ]:
import re
import time
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

START_URL = "https://books.toscrape.com/catalogue/page-1.html"

# How many pages to scrape:
#   3    -> minimum requirement
#   5    -> Bonus 1
#   None -> Boss Level (every catalogue page, follows the "next" button until it disappears)
MAX_PAGES = None

DELAY_SECONDS = 0.5   # polite pause between requests
HEADERS = {"User-Agent": "Mozilla/5.0 (student web-scraping assignment)"}
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

pd.set_option("display.max_colwidth", 60)

## Reusable scraping functions

All scraping logic lives in these functions, so page 1, three pages, five pages and the whole catalogue all use **exactly the same code**.

In [ ]:
def get_soup(url):
    """Download a page and return it as a BeautifulSoup object."""
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.raise_for_status()
    # Pass raw bytes so BeautifulSoup reads the page's UTF-8 charset (keeps the £ sign intact)
    return BeautifulSoup(response.content, "html.parser")


def parse_book(card, page_url):
    """Extract one book's fields from an <article class='product_pod'> card."""
    link = card.h3.a

    title = link["title"].strip()

    price_text = card.find("p", class_="price_color").get_text(strip=True)   # e.g. '£51.77'
    price = float(re.search(r"\d+(?:\.\d+)?", price_text).group())

    rating_classes = card.find("p", class_="star-rating")["class"]           # e.g. ['star-rating', 'Three']
    rating_word = next((c for c in rating_classes if c in RATING_MAP), None)
    rating = RATING_MAP.get(rating_word)

    availability = card.find("p", class_="availability").get_text(strip=True)

    url = urljoin(page_url, link["href"])                                     # Bonus 2

    return {"Title": title, "Price": price, "Rating": rating,
            "Availability": availability, "URL": url}


def scrape_page(page_url):
    """Scrape every book on one catalogue page. Returns (list_of_books, next_page_url_or_None)."""
    soup = get_soup(page_url)
    books = [parse_book(card, page_url) for card in soup.find_all("article", class_="product_pod")]

    next_link = soup.select_one("li.next > a")
    next_url = urljoin(page_url, next_link["href"]) if next_link else None
    return books, next_url


def scrape_catalogue(start_url, max_pages=None, delay=DELAY_SECONDS):
    """Loop through catalogue pages by following the 'next' button."""
    all_books, url, page = [], start_url, 0
    while url and (max_pages is None or page < max_pages):
        page += 1
        books, url = scrape_page(url)
        all_books.extend(books)
        print(f"Page {page:>2}: {len(books)} books scraped  (running total: {len(all_books)})")
        time.sleep(delay)
    return pd.DataFrame(all_books)

## Mission 2 — Capture the Data (first page)

In [ ]:
first_page_books, _ = scrape_page(START_URL)
df_page1 = pd.DataFrame(first_page_books)

print(f"Books found on page 1: {len(df_page1)}")
df_page1[["Title", "Price", "Rating", "Availability"]]

## Mission 3 — Scale the Operation

The same functions run inside a loop. Change `MAX_PAGES` in the setup cell to 3, 5 or `None` (all pages).

In [ ]:
df = scrape_catalogue(START_URL, max_pages=MAX_PAGES)
print(f"\nRaw records scraped: {len(df)}")

### Clean, check and save the dataset

In [ ]:
# Basic cleaning
df["Title"] = df["Title"].str.strip()
df["Availability"] = df["Availability"].str.strip()
df = df.drop_duplicates(subset="URL").reset_index(drop=True)   # one row per product page
df["Price"] = df["Price"].astype(float)
df["Rating"] = df["Rating"].astype(int)

# Quality checks
print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes, sep="")
print("\nMissing values per column:\n", df.isna().sum(), sep="")
assert list(df.columns[:4]) == ["Title", "Price", "Rating", "Availability"], "Unexpected columns"
assert df["Rating"].between(1, 5).all(), "Ratings must be 1-5"

df.to_csv("competitor_books.csv", index=False, encoding="utf-8")
print("\nSaved competitor_books.csv")

df.head()

## Mission 4 — Intelligence Analysis

In [ ]:
def show(rows):
    return rows[["Title", "Price", "Rating", "Availability"]].to_string(index=False)

max_price, min_price = df["Price"].max(), df["Price"].min()
most_expensive = df[df["Price"] == max_price]
cheapest       = df[df["Price"] == min_price]
avg_price      = df["Price"].mean()
five_star      = df[df["Rating"] == 5]
pct_in_stock   = df["Availability"].str.contains("In stock", case=False).mean() * 100
# Many books tie on rating, so ties are broken by lower price (best value for the customer)
top5           = df.sort_values(["Rating", "Price"], ascending=[False, True]).head(5)
cheapest_5star = five_star[five_star["Price"] == five_star["Price"].min()]

print(f"Dataset analysed: {len(df)} books\n")
print("Q1. Most expensive book:\n" + show(most_expensive) + "\n")
print("Q2. Cheapest book:\n" + show(cheapest) + "\n")
print(f"Q3. Average book price: £{avg_price:.2f}\n")
print(f"Q4. Number of 5-star books: {len(five_star)} ({len(five_star)/len(df)*100:.1f}% of the dataset)\n")
print(f"Q5. Books in stock: {pct_in_stock:.1f}%\n")
print(f"Q6. Top 5 highest-rated books (ties broken by lower price):\n" + show(top5) + "\n")
print("Q7. Cheapest 5-star book:\n" + show(cheapest_5star))

### Extra evidence for the business decision

In [ ]:
summary = (df.groupby("Rating")
             .agg(Books=("Title", "count"),
                  Avg_Price=("Price", "mean"),
                  Min_Price=("Price", "min"),
                  Max_Price=("Price", "max"))
             .round(2))
print("Price profile by star rating:\n", summary, sep="")

price_bands = pd.cut(df["Price"], bins=[0, 20, 30, 40, 50, 60], labels=["£10-20", "£20-30", "£30-40", "£40-50", "£50-60"])
print("\nBooks per price band:\n", price_bands.value_counts().sort_index(), sep="")

value_picks = five_star[five_star["Price"] < avg_price].sort_values("Price")
print(f"\n5-star books priced below the average (£{avg_price:.2f}): {len(value_picks)}")
print(show(value_picks.head(10)))

## Final Challenge — The CEO Has £100

This is a **0/1 knapsack problem**: each book can be bought once, the budget is £100, and we want the highest total star rating.
Prices are converted to pence so the calculation is exact. Dynamic programming checks every combination efficiently; if two baskets
reach the same star total, the cheaper one is chosen.

In [ ]:
BUDGET = 100.00

def best_basket(data, budget=BUDGET):
    data = data.reset_index(drop=True)
    costs  = (data["Price"] * 100).round().astype(int).to_numpy()
    values = data["Rating"].to_numpy()
    cap = int(round(budget * 100))

    dp = np.zeros(cap + 1, dtype=np.int64)            # dp[c] = best rating with spend <= c pence
    take = np.zeros((len(data), cap + 1), dtype=bool)  # remembers decisions for backtracking

    for i, (c, v) in enumerate(zip(costs, values)):
        if c > cap:
            continue
        candidate = dp[:cap + 1 - c] + v
        better = candidate > dp[c:]
        take[i, c:] = better
        dp[c:] = np.where(better, candidate, dp[c:])

    best_score = dp[cap]
    spend = int(np.argmax(dp == best_score))          # cheapest spend that still reaches the best score
    chosen = []
    for i in range(len(data) - 1, -1, -1):
        if take[i, spend]:
            chosen.append(i)
            spend -= costs[i]
    return data.loc[sorted(chosen)].sort_values("Price")

basket = best_basket(df)

print("CEO £100 MARKET-RESEARCH BASKET")
print(basket[["Title", "Price", "Rating"]].to_string(index=False))
print(f"\nBooks bought:      {len(basket)}")
print(f"Total cost:        £{basket['Price'].sum():.2f}  (budget £{BUDGET:.2f}, left over £{BUDGET - basket['Price'].sum():.2f})")
print(f"Total star rating: {basket['Rating'].sum()}")

## Bonus 3 & 4 — Rating counts and bar chart

In [ ]:
rating_counts = df["Rating"].value_counts().sort_index()
print("Books per star rating:\n", rating_counts.to_string(), sep="")

ax = rating_counts.plot(kind="bar", color="#2E86AB", edgecolor="black", figsize=(7, 4))
ax.set_title(f"Competitor Books by Star Rating (n = {len(df)})")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of books")
ax.tick_params(axis="x", rotation=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig("rating_counts.png", dpi=150)
plt.show()

## Download the files

In [ ]:
try:
    from google.colab import files
    files.download("competitor_books.csv")
    files.download("rating_counts.png")
except ImportError:
    print("Not running on Colab: competitor_books.csv and rating_counts.png are saved in the working folder.")